# 08 딥러닝 예측 모델

LSTM, Autoformer, N-HiTS, iTransformer를 **type×family** 주간 시계열에 적용합니다 (CPU, 논문 설정 유지).

- 검증 구간: 201731–201733

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX
from utils.metrics import wmape
from utils.dl_models import DL_MODELS

df = pd.read_parquet(DATA_PROCESSED / 'df_weekly.parquet')
df = df.sort_values(['type', 'family', 'yearweek']).reset_index(drop=True)
HORIZON = len(VAL_WEEKS)
LOOKBACK = 16
print('series:', df.groupby(['type','family']).ngroups, '| horizon:', HORIZON)

In [ ]:
rows = []
for (typ, fam), g in tqdm(df.groupby(['type', 'family']), desc='DL forecast'):
    g_train = g[g['yearweek'] <= TRAIN_WEEK_MAX]
    g_val = g[g['yearweek'].isin(VAL_WEEKS)].sort_values('yearweek')
    if len(g_val) == 0:
        continue
    y_true = g_val['sales'].values.astype(float)
    series = g_train['sales'].values.astype(float)

    for name, fn in DL_MODELS.items():
        pred = fn(LOOKBACK, HORIZON, series)
        if len(pred) != len(y_true):
            pred = np.resize(pred, len(y_true))
        rows.append({
            'type': typ, 'family': fam, 'model': name,
            'wmape': wmape(y_true, pred),
            'mae': float(np.mean(np.abs(y_true - pred))),
        })

dl_results = pd.DataFrame(rows)
summary = dl_results.groupby('model')['wmape'].mean().sort_values().reset_index()
print('=== DL 모델별 평균 WMAPE (%) ===')
print(summary.round(2))

out = DATA_PROCESSED / 'forecast_dl_results.parquet'
dl_results.to_parquet(out, index=False)
summary.to_csv(DATA_PROCESSED / 'forecast_dl_summary.csv', index=False)
print('저장:', out)
dl_results.head()